# Chunked Cambridge House Price Analysis

This notebook analyzes Cambridge HM Land Registry price paid data from `ppd_data.csv`.

It is designed to be memory-safe for large files:
- Reads the CSV in chunks
- Maintains running aggregations instead of concatenating all rows
- Uses fixed-bin histograms for approximate medians/quantiles
- Keeps only a capped sample for distribution/scatter-style plots

In [ ]:
# Install required dependencies if they are missing.
import importlib
import subprocess
import sys

REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
}

missing = []
for module_name, package_name in REQUIRED_PACKAGES.items():
    try:
        importlib.import_module(module_name)
    except ImportError:
        missing.append(package_name)

if missing:
    print(f"Installing missing packages: {missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("All required packages are already installed.")

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from collections import defaultdict

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)

DATA_PATH = "ppd_data.csv"
CHUNK_SIZE = 50_000
SAMPLE_CAP = 20_000

print(f"Using chunk size: {CHUNK_SIZE:,}")
print(f"Sample cap for plot-level rows: {SAMPLE_CAP:,}")

ModuleNotFoundError: No module named 'numpy'

In [ ]:
# PPD CSV has no header row; define columns explicitly.
PPD_COLUMNS = [
    "transaction_id",
    "price",
    "date",
    "postcode",
    "property_type",
    "new_build",
    "tenure",
    "paon",
    "saon",
    "street",
    "locality",
    "town_city",
    "district",
    "county",
    "ppd_category_type",
    "record_status",
]

USECOLS = [
    "transaction_id",
    "price",
    "date",
    "postcode",
    "property_type",
    "new_build",
    "tenure",
    "district",
    "county",
    "record_status",
]

PROPERTY_TYPE_MAP = {
    "D": "Detached",
    "S": "Semi-detached",
    "T": "Terraced",
    "F": "Flat/Maisonette",
    "O": "Other",
}

TENURE_MAP = {"F": "Freehold", "L": "Leasehold"}
NEW_BUILD_MAP = {"Y": "New build", "N": "Existing"}

# Log10(price) bins for memory-efficient approximate quantiles.
LOG_BIN_EDGES = np.linspace(4.0, 7.2, 180)  # ~10k to ~16m
N_BINS = len(LOG_BIN_EDGES) - 1


def prices_to_hist(prices: pd.Series) -> np.ndarray:
    clipped = prices.clip(lower=10_000, upper=16_000_000)
    return np.histogram(np.log10(clipped), bins=LOG_BIN_EDGES)[0]


def approx_quantiles_from_hist(hist_counts: np.ndarray, quantiles=(0.25, 0.5, 0.75, 0.95, 0.99)) -> dict:
    total = hist_counts.sum()
    if total == 0:
        return {q: np.nan for q in quantiles}

    cdf = np.cumsum(hist_counts) / total
    out = {}
    for q in quantiles:
        idx = int(np.searchsorted(cdf, q, side="left"))
        idx = max(0, min(idx, N_BINS - 1))
        log_price = 0.5 * (LOG_BIN_EDGES[idx] + LOG_BIN_EDGES[idx + 1])
        out[q] = float(10 ** log_price)
    return out

In [ ]:
# Running aggregations (memory-safe)
rows_seen = 0
rows_valid = 0
invalid_price_rows = 0
invalid_date_rows = 0
unexpected_property_type_rows = 0
nonstandard_record_status_rows = 0
duplicate_txn_within_chunk = 0

min_date = None
max_date = None

overall_count = 0
overall_sum = 0.0
overall_sum_sq = 0.0
overall_min = np.inf
overall_max = 0.0
overall_hist = np.zeros(N_BINS, dtype=np.int64)

by_property = defaultdict(lambda: {"count": 0, "sum": 0.0, "hist": np.zeros(N_BINS, dtype=np.int64)})
by_tenure = defaultdict(lambda: {"count": 0, "sum": 0.0, "hist": np.zeros(N_BINS, dtype=np.int64)})
by_new_build = defaultdict(lambda: {"count": 0, "sum": 0.0, "hist": np.zeros(N_BINS, dtype=np.int64)})

by_month = defaultdict(lambda: {"count": 0, "sum": 0.0, "hist": np.zeros(N_BINS, dtype=np.int64)})
by_year = defaultdict(lambda: {"count": 0, "sum": 0.0, "hist": np.zeros(N_BINS, dtype=np.int64)})

by_postcode_district = defaultdict(lambda: {"count": 0, "sum": 0.0, "hist": np.zeros(N_BINS, dtype=np.int64)})

sample_parts = []

reader = pd.read_csv(
    DATA_PATH,
    header=None,
    names=PPD_COLUMNS,
    usecols=USECOLS,
    parse_dates=["date"],
    chunksize=CHUNK_SIZE,
    low_memory=True,
)

for chunk in reader:
    rows_seen += len(chunk)

    # Basic quality checks and cleanup.
    duplicate_txn_within_chunk += int(chunk["transaction_id"].duplicated().sum())
    nonstandard_record_status_rows += int((chunk["record_status"] != "A").sum())

    chunk["price"] = pd.to_numeric(chunk["price"], errors="coerce")
    invalid_price_rows += int(chunk["price"].isna().sum())

    invalid_date_rows += int(chunk["date"].isna().sum())

    chunk = chunk.dropna(subset=["price", "date"]) 
    chunk = chunk[chunk["price"] > 0]

    valid_property_mask = chunk["property_type"].isin(PROPERTY_TYPE_MAP)
    unexpected_property_type_rows += int((~valid_property_mask).sum())
    chunk = chunk[valid_property_mask]

    if chunk.empty:
        continue

    chunk = chunk.copy()
    chunk["month"] = chunk["date"].dt.to_period("M").astype(str)
    chunk["year"] = chunk["date"].dt.year.astype(int)
    chunk["postcode_district"] = chunk["postcode"].fillna("").str.split().str[0]

    rows_valid += len(chunk)

    # Update overall aggregations.
    prices = chunk["price"]
    overall_count += len(chunk)
    overall_sum += float(prices.sum())
    overall_sum_sq += float((prices ** 2).sum())
    overall_min = min(overall_min, float(prices.min()))
    overall_max = max(overall_max, float(prices.max()))
    overall_hist += prices_to_hist(prices)

    cmin = chunk["date"].min()
    cmax = chunk["date"].max()
    min_date = cmin if min_date is None else min(min_date, cmin)
    max_date = cmax if max_date is None else max(max_date, cmax)

    # Group-level aggregations.
    for key, grp in chunk.groupby("property_type"):
        by_property[key]["count"] += len(grp)
        by_property[key]["sum"] += float(grp["price"].sum())
        by_property[key]["hist"] += prices_to_hist(grp["price"])

    for key, grp in chunk.groupby("tenure"):
        by_tenure[key]["count"] += len(grp)
        by_tenure[key]["sum"] += float(grp["price"].sum())
        by_tenure[key]["hist"] += prices_to_hist(grp["price"])

    for key, grp in chunk.groupby("new_build"):
        by_new_build[key]["count"] += len(grp)
        by_new_build[key]["sum"] += float(grp["price"].sum())
        by_new_build[key]["hist"] += prices_to_hist(grp["price"])

    for key, grp in chunk.groupby("month"):
        by_month[key]["count"] += len(grp)
        by_month[key]["sum"] += float(grp["price"].sum())
        by_month[key]["hist"] += prices_to_hist(grp["price"])

    for key, grp in chunk.groupby("year"):
        by_year[key]["count"] += len(grp)
        by_year[key]["sum"] += float(grp["price"].sum())
        by_year[key]["hist"] += prices_to_hist(grp["price"])

    for key, grp in chunk.groupby("postcode_district"):
        by_postcode_district[key]["count"] += len(grp)
        by_postcode_district[key]["sum"] += float(grp["price"].sum())
        by_postcode_district[key]["hist"] += prices_to_hist(grp["price"])

    # Keep a capped random sample for selected visualizations.
    sample_chunk = chunk[["date", "price", "property_type", "tenure", "new_build", "postcode_district"]]
    sample_chunk = sample_chunk.sample(n=min(1500, len(sample_chunk)), random_state=42)
    sample_parts.append(sample_chunk)

sample_df = pd.concat(sample_parts, ignore_index=True) if sample_parts else pd.DataFrame()
if len(sample_df) > SAMPLE_CAP:
    sample_df = sample_df.sample(n=SAMPLE_CAP, random_state=42).sort_values("date")

print(f"Rows seen: {rows_seen:,}")
print(f"Rows valid for analysis: {rows_valid:,}")
print(f"Sampled rows kept for plotting: {len(sample_df):,}")

In [ ]:
overall_quantiles = approx_quantiles_from_hist(overall_hist, quantiles=(0.25, 0.5, 0.75, 0.95, 0.99))
overall_mean = overall_sum / overall_count if overall_count else np.nan
overall_var = (overall_sum_sq / overall_count) - (overall_mean ** 2) if overall_count else np.nan
overall_std = np.sqrt(max(0.0, overall_var)) if overall_count else np.nan

overall_stats = pd.DataFrame(
    {
        "metric": [
            "rows_seen",
            "rows_valid",
            "date_min",
            "date_max",
            "price_mean",
            "price_std",
            "price_min",
            "price_q25_approx",
            "price_median_approx",
            "price_q75_approx",
            "price_q95_approx",
            "price_q99_approx",
            "price_max",
        ],
        "value": [
            rows_seen,
            rows_valid,
            min_date,
            max_date,
            overall_mean,
            overall_std,
            overall_min,
            overall_quantiles[0.25],
            overall_quantiles[0.5],
            overall_quantiles[0.75],
            overall_quantiles[0.95],
            overall_quantiles[0.99],
            overall_max,
        ],
    }
)

overall_stats

In [ ]:
def grouped_summary_from_tracker(tracker: dict, label_map: dict | None = None, group_col: str = "group") -> pd.DataFrame:
    rows = []
    for group_key, values in tracker.items():
        count = values["count"]
        if count == 0:
            continue
        quantiles = approx_quantiles_from_hist(values["hist"], quantiles=(0.25, 0.5, 0.75))
        rows.append(
            {
                group_col: label_map.get(group_key, group_key) if label_map else group_key,
                "count": count,
                "mean_price": values["sum"] / count,
                "median_price_approx": quantiles[0.5],
                "q25_price_approx": quantiles[0.25],
                "q75_price_approx": quantiles[0.75],
            }
        )
    if not rows:
        return pd.DataFrame(columns=[group_col, "count", "mean_price", "median_price_approx", "q25_price_approx", "q75_price_approx"])
    return pd.DataFrame(rows).sort_values("count", ascending=False).reset_index(drop=True)


property_summary = grouped_summary_from_tracker(by_property, PROPERTY_TYPE_MAP, group_col="property_type")
tenure_summary = grouped_summary_from_tracker(by_tenure, TENURE_MAP, group_col="tenure")
new_build_summary = grouped_summary_from_tracker(by_new_build, NEW_BUILD_MAP, group_col="build_status")

property_summary, tenure_summary, new_build_summary

In [ ]:
monthly_summary = grouped_summary_from_tracker(by_month, group_col="month")
if not monthly_summary.empty:
    monthly_summary = monthly_summary.sort_values("month").reset_index(drop=True)
    monthly_summary["month"] = pd.to_datetime(monthly_summary["month"])
    monthly_summary["rolling_median_6m"] = monthly_summary["median_price_approx"].rolling(6, min_periods=3).mean()

yearly_summary = grouped_summary_from_tracker(by_year, group_col="year")
if not yearly_summary.empty:
    yearly_summary = yearly_summary.sort_values("year").reset_index(drop=True)

postcode_summary = grouped_summary_from_tracker(by_postcode_district, group_col="postcode_district")
postcode_summary = postcode_summary[postcode_summary["postcode_district"].str.startswith("CB", na=False)]

monthly_summary.head(), yearly_summary.head(), postcode_summary.head()

## Plot 1: Monthly median price trend (approx)

This uses histogram-based approximate medians computed chunk-by-chunk.

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(monthly_summary["month"], monthly_summary["median_price_approx"], label="Monthly median (approx)", linewidth=1.8)
if "rolling_median_6m" in monthly_summary:
    plt.plot(monthly_summary["month"], monthly_summary["rolling_median_6m"], label="6-month rolling median", linewidth=2.2)
plt.title("Cambridge sold-price trend")
plt.xlabel("Month")
plt.ylabel("Price (GBP)")
plt.legend()
plt.tight_layout()
plt.show()

Look for long-run trend direction and periods of temporary corrections. Rolling median helps reduce month-to-month noise from compositional shifts.

## Plot 2: Monthly transaction volume

In [ ]:
plt.figure(figsize=(12, 4.5))
plt.bar(monthly_summary["month"], monthly_summary["count"], width=25)
plt.title("Monthly transaction volume")
plt.xlabel("Month")
plt.ylabel("Number of sales")
plt.tight_layout()
plt.show()

Compare peaks/troughs in activity with price changes to spot potential lag effects between demand and observed transaction prices.

## Plot 3: Price distribution (log scale)

In [ ]:
bin_centers = 0.5 * (LOG_BIN_EDGES[:-1] + LOG_BIN_EDGES[1:])
plt.figure(figsize=(10, 4.5))
plt.bar(bin_centers, overall_hist, width=np.diff(LOG_BIN_EDGES), align="center")
plt.title("Distribution of sold prices (log10 scale)")
plt.xlabel("log10(price)")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

Log scaling is useful because house prices are right-skewed; it reveals the modal band more clearly than raw-price histograms.

## Plot 4: Property type comparison (sampled rows)

In [ ]:
if not sample_df.empty:
    plot_df = sample_df.copy()
    plot_df["property_type_label"] = plot_df["property_type"].map(PROPERTY_TYPE_MAP).fillna("Other")
    ymax = plot_df["price"].quantile(0.98)

    plt.figure(figsize=(11, 5))
    sns.boxplot(data=plot_df, x="property_type_label", y="price", showfliers=False)
    plt.ylim(0, ymax)
    plt.title("Price by property type (sample, clipped at 98th percentile)")
    plt.xlabel("Property type")
    plt.ylabel("Price (GBP)")
    plt.tight_layout()
    plt.show()
else:
    print("No sample rows available for property type plot.")

This plot highlights spread differences by property type; clipping at the high tail improves readability for the bulk of the market.

## Plot 5: Tenure comparison (sampled rows)

In [ ]:
if not sample_df.empty:
    plot_df = sample_df.copy()
    plot_df["tenure_label"] = plot_df["tenure"].map(TENURE_MAP).fillna("Unknown")
    ymax = plot_df["price"].quantile(0.98)

    plt.figure(figsize=(8, 5))
    sns.boxplot(data=plot_df, x="tenure_label", y="price", showfliers=False)
    plt.ylim(0, ymax)
    plt.title("Price by tenure (sample, clipped at 98th percentile)")
    plt.xlabel("Tenure")
    plt.ylabel("Price (GBP)")
    plt.tight_layout()
    plt.show()
else:
    print("No sample rows available for tenure plot.")

Tenure composition can shift through time, so compare this with the monthly trend chart before drawing structural conclusions.

## Plot 6: Top Cambridge postcode districts

In [ ]:
top_n = 10
plot_postcodes = postcode_summary.nlargest(top_n, "count").sort_values("median_price_approx", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=plot_postcodes, x="median_price_approx", y="postcode_district", ax=axes[0], orient="h")
axes[0].set_title(f"Top {top_n} districts by median price (approx)")
axes[0].set_xlabel("Median price (GBP)")
axes[0].set_ylabel("Postcode district")

sns.barplot(data=plot_postcodes, x="count", y="postcode_district", ax=axes[1], orient="h")
axes[1].set_title(f"Top {top_n} districts by transaction count")
axes[1].set_xlabel("Transaction count")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()

District-level medians should be interpreted with volume in mind; low-count districts can show unstable median estimates.

## Optional plot: Date vs price scatter (sampled)

In [ ]:
if not sample_df.empty:
    plt.figure(figsize=(12, 5))
    plt.scatter(sample_df["date"], sample_df["price"], s=8, alpha=0.2)
    plt.title("Sampled transactions: date vs sold price")
    plt.xlabel("Date")
    plt.ylabel("Price (GBP)")
    plt.tight_layout()
    plt.show()
else:
    print("No sample rows available for scatter plot.")

## Data quality and processing checks

In [ ]:
quality_checks = pd.DataFrame(
    {
        "check": [
            "rows_seen",
            "rows_valid",
            "invalid_price_rows",
            "invalid_date_rows",
            "unexpected_property_type_rows",
            "nonstandard_record_status_rows",
            "duplicate_txn_ids_within_chunk",
            "overall_count_matches_rows_valid",
        ],
        "value": [
            rows_seen,
            rows_valid,
            invalid_price_rows,
            invalid_date_rows,
            unexpected_property_type_rows,
            nonstandard_record_status_rows,
            duplicate_txn_within_chunk,
            overall_count == rows_valid,
        ],
    }
)

quality_checks

## Key summary tables

In [ ]:
display(overall_stats)
display(property_summary)
display(tenure_summary)
display(new_build_summary)
display(yearly_summary.tail(15))

## Suggested next plots

If you want to extend this notebook, these are high-value additions:
1. **Inflation-adjusted trend** (deflate nominal prices using CPI index).
2. **Repeat-sales proxy** by matching likely same property address tokens.
3. **Volatility over time** (rolling IQR / rolling coefficient of variation).
4. **Price per postcode district over time** (small multiples for `CB1`, `CB2`, ...).
5. **Segmented trend** by property type and tenure to separate composition effects.